# DBTS Variants + Regime Classifier — edge test

Pipeline: **NS_spline regressor → DBTS selector (rz_x_q | all | soft, winner of DBTS_Variants) → XGBoost regime classifier → PM**.

Reuses caches from `outputs/dbts_variants/` (NS regressors + quality).

Compare 4 PM modes across TRAIN / VAL / TEST:
- **A) NoCLF** — baseline, pure mean-rev on |rz|≥1.5 (= DBTS_Variants winner).
- **B) MR_GATE** — enter ONLY if classifier predicts MR and `conf ≥ conf_thr`. Direction = fade.
- **C) NotNEUTRAL** — enter if classifier predicts MR or MOM (not Neutral) and `conf ≥ conf_thr`. Direction = fade (we ignore MOM-specific logic; pure mean-rev with regime confirmation).
- **D) DUAL** — MR → fade; MOM → follow (regime-aware direction, like DBTS_NoBandit).

Sweep `conf_thr ∈ {0.35, 0.40, 0.45, 0.55}` for B/C/D.

Decision: if **VAL Sharpe of best CLF mode > VAL Sharpe of NoCLF by ≥ 0.10**, adopt classifier; else keep DBTS_Variants.

In [ ]:
# Cell 2 — Imports + setup
import os, sys, math, pickle, warnings, time
import random as _random
from pathlib import Path
from datetime import datetime
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings('ignore')
%matplotlib inline
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists(): PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT)); os.chdir(PROJECT_ROOT)
SEED = 42
_random.seed(SEED); np.random.seed(SEED)
FORCE_RECOMPUTE = False
SRC_CACHE = Path('outputs/dbts_variants')      # reuse NS regressors + quality from DBTS_Variants
DST_CACHE = Path('outputs/dbts_variants_clf'); DST_CACHE.mkdir(parents=True, exist_ok=True)
REG_CACHE     = SRC_CACHE / 'ns_regressors.pkl'
QUALITY_CACHE = SRC_CACHE / 'quality.pkl'
TECH_CACHE    = DST_CACHE / 'tech_panel.pkl'
CLF_CACHE     = DST_CACHE / 'regime_clf.pkl'
print(f'src cache: {SRC_CACHE}\ndst cache: {DST_CACHE}')

In [ ]:
# Cell 3 — Data, splits, sector pools (same as DBTS_Variants)
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split
cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)
md = pipeline.load_data()
sp = chrono_split(md.prices.index, cfg)
train_idx = pd.DatetimeIndex(sp.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(sp.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(sp.test_idx).sort_values()
SECTOR_POOLS = {}
for etf, s in SECTORS.items():
    pool = [s['target']] + list(s['predictors'])
    pool = [t for t in pool if t in md.prices.columns]
    SECTOR_POOLS[s['name']] = pool
ALL_TICKERS = sorted({t for pool in SECTOR_POOLS.values() for t in pool})
print(f'TRAIN n={len(train_idx)} | VAL n={len(val_idx)} | TEST n={len(test_idx)}')
print(f'sectors: {len(SECTOR_POOLS)}, total candidates: {sum(len(p) for p in SECTOR_POOLS.values())}, unique tickers: {len(ALL_TICKERS)}')

In [ ]:
# Cell 4 — Load cached NS regressors + quality (must exist)
assert REG_CACHE.exists(), f'Missing {REG_CACHE} — run DBTS_Variants first.'
assert QUALITY_CACHE.exists(), f'Missing {QUALITY_CACHE} — run DBTS_Variants first.'
with open(REG_CACHE,'rb') as f:     REG  = pickle.load(f)
with open(QUALITY_CACHE,'rb') as f: QUAL = pickle.load(f)
print(f'REG: {len(REG)} pairs | QUAL: {len(QUAL)} pairs')

In [ ]:
# Cell 5 — Tech features per ticker + regime labels (copied from DBTS_NoBandit_EN_Regime)
LABEL_H = 5; VOL_WIN = 60
_TO = {-1:0, 0:1, 1:2}; _FROM = {0:-1, 1:0, 2:1}

def make_regime_label(close, h=LABEL_H, vol_win=VOL_WIN):
    ret_h = close.pct_change(h).shift(-h)
    vol   = close.pct_change().rolling(vol_win).std() * math.sqrt(h)
    z = ret_h / vol
    lab = pd.Series(0, index=close.index, dtype=float)
    lab[z >  0.5] =  1
    lab[z < -0.5] = -1
    lab[z.isna()] = np.nan
    return lab

def make_tech_features(close):
    df = pd.DataFrame(index=close.index)
    ma20 = close.rolling(20).mean(); ma50 = close.rolling(50).mean(); ma200 = close.rolling(200).mean()
    df['ma_short_above_long']  = (ma20 > ma50).astype(int)
    df['ma_med_above_long']    = (ma50 > ma200).astype(int)
    df['golden_cross_recent']  = ((ma50 > ma200) & (ma50.shift(20) <= ma200.shift(20))).astype(int)
    df['death_cross_recent']   = ((ma50 < ma200) & (ma50.shift(20) >= ma200.shift(20))).astype(int)
    df['price_above_ma50']     = (close > ma50).astype(int)
    df['price_above_ma200']    = (close > ma200).astype(int)
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan); rsi = 100 - 100/(1+rs)
    df['rsi_overbought']       = (rsi > 70).astype(int)
    df['rsi_oversold']         = (rsi < 30).astype(int)
    bb_mid = ma20; bb_std = close.rolling(20).std(); bb_up = bb_mid + 2*bb_std; bb_lo = bb_mid - 2*bb_std
    df['bb_upper_breach']      = (close > bb_up).astype(int)
    df['bb_lower_breach']      = (close < bb_lo).astype(int)
    hi20 = close.rolling(20).max(); lo20 = close.rolling(20).min()
    df['breakout_20d_high']    = (close >= hi20).astype(int)
    df['breakdown_20d_low']    = (close <= lo20).astype(int)
    rng = close.pct_change().rolling(5).std()
    rng_med = rng.rolling(60).median()
    df['vol_spike']            = (rng > 1.5 * rng_med).astype(int)
    ud = (delta > 0).astype(int) - (delta < 0).astype(int)
    df['consec_up_3d']         = ((ud.rolling(3).sum()) == 3).astype(int)
    df['consec_down_3d']       = ((ud.rolling(3).sum()) == -3).astype(int)
    df['narrow_range']         = (rng < 0.5 * rng_med).astype(int)
    return df.fillna(0).astype(int)

if TECH_CACHE.exists() and not FORCE_RECOMPUTE:
    tech_panel = pd.read_pickle(TECH_CACHE)
    print(f'[CACHE] tech_panel: {len(tech_panel):,} rows')
else:
    print(f'[RECOMPUTE] tech for {len(ALL_TICKERS)} tickers...')
    rows = []
    for t in ALL_TICKERS:
        if t not in md.prices.columns: continue
        f = make_tech_features(md.prices[t])
        f['date'] = f.index; f['ticker'] = t
        rows.append(f.reset_index(drop=True))
    tech_panel = pd.concat(rows, ignore_index=True)
    tech_panel.to_pickle(TECH_CACHE)
    print(f'Saved -> {TECH_CACHE.name}, {len(tech_panel):,} rows')
TECH_COLS = [c for c in tech_panel.columns if c not in ('date','ticker')]
tech_panel['date'] = pd.to_datetime(tech_panel['date'])
print(f'Feature cols ({len(TECH_COLS)}): {TECH_COLS}')

In [ ]:
# Cell 6 — Train XGB regime classifier on TRAIN, evaluate on VAL
if CLF_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(CLF_CACHE,'rb') as f: blob = pickle.load(f)
    clf = blob['model']; TECH_COLS = blob['features']
    print('[CACHE] regime_clf loaded')
else:
    print('[RECOMPUTE] labels + train XGB...')
    label_rows = []
    for t in ALL_TICKERS:
        if t not in md.prices.columns: continue
        lab = make_regime_label(md.prices[t])
        label_rows.append(pd.DataFrame({'date': lab.index, 'ticker': t, 'regime_label': lab.values}))
    labels = pd.concat(label_rows, ignore_index=True)
    panel = tech_panel.merge(labels, on=['date','ticker'], how='left')
    panel['date'] = pd.to_datetime(panel['date'])
    data_tr = panel[panel['date'].isin(train_idx) & panel['regime_label'].notna()]
    data_va = panel[panel['date'].isin(val_idx)   & panel['regime_label'].notna()]
    X_tr = data_tr[TECH_COLS].astype(float); y_tr = data_tr['regime_label'].map(_TO).astype(int)
    X_va = data_va[TECH_COLS].astype(float); y_va = data_va['regime_label'].map(_TO).astype(int)
    sw = compute_sample_weight(class_weight='balanced', y=y_tr)
    clf = XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3,
                        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.1,
                        num_class=3, objective='multi:softprob', verbosity=0,
                        random_state=SEED, seed=SEED, n_jobs=1, tree_method='exact')
    clf.fit(X_tr, y_tr, sample_weight=sw)
    pred_va = clf.predict(X_va)
    print(f'\nVAL acc={accuracy_score(y_va, pred_va):.4f} f1_macro={f1_score(y_va, pred_va, average="macro"):.4f}')
    print('Confusion (rows=true, cols=pred):')
    print(pd.DataFrame(confusion_matrix(y_va, pred_va, labels=[0,1,2]),
                       index=['MR','N','MOM'], columns=['MR','N','MOM']))
    with open(CLF_CACHE,'wb') as f: pickle.dump({'model': clf, 'features': TECH_COLS}, f)
    print(f'Saved -> {CLF_CACHE.name}')

# predict on FULL tech panel
X_all = tech_panel[TECH_COLS].astype(float)
proba = clf.predict_proba(X_all)
tech_panel['regime_pred'] = np.array([_FROM[i] for i in clf.predict(X_all)])
tech_panel['P_MR']        = proba[:,0]
tech_panel['P_NEUTRAL']   = proba[:,1]
tech_panel['P_MOM']       = proba[:,2]
tech_panel['regime_conf'] = proba.max(axis=1)
regm_idx = tech_panel.set_index(['date','ticker'])[['regime_pred','regime_conf','P_MR','P_NEUTRAL','P_MOM']]
print(f'\nRegime distribution (full sample): {dict(zip(*np.unique(tech_panel["regime_pred"], return_counts=True)))}')

In [ ]:
# Cell 7 — Build winner panel (rz_x_q | all | soft) WITH regime predictions merged in
STACKS = {}
for sector, pool in SECTOR_POOLS.items():
    cands = [c for c in pool if (sector, c) in REG]
    if not cands: continue
    idx = md.prices.index
    resz = pd.DataFrame({c: REG[(sector,c)]['residual_z']       for c in cands}).reindex(idx)
    pr   = pd.DataFrame({c: REG[(sector,c)]['predicted_return'] for c in cands}).reindex(idx)
    nxt  = pd.DataFrame({c: REG[(sector,c)]['next_ret']         for c in cands}).reindex(idx)
    qv   = np.array([QUAL[(sector,c)]['quality'] for c in cands])
    STACKS[sector] = dict(cands=cands, resz=resz, pr=pr, nxt=nxt, quality=qv)

def build_winner_panel():
    """score=rz_x_q, universe=all, stick=soft (min_hold=5, margin=+0.10 abs)."""
    rows = []
    for sector, st in STACKS.items():
        cands = st['cands']; idx = st['resz'].index
        cand_arr = np.array(cands)
        resz = st['resz'].values; pr = st['pr'].values; nxt = st['nxt'].values
        qv = st['quality']
        abs_rz = np.abs(np.nan_to_num(resz, nan=0.0))
        scores = abs_rz * qv[None, :]
        valid = (~np.isnan(resz))
        scores = np.where(valid, scores, -np.inf)
        n = len(idx); win = np.empty(n, dtype=int); win[:] = -1
        cur = -1; held = 0
        for i in range(n):
            ds = scores[i]
            if not np.any(np.isfinite(ds)): cur=-1; held=0; continue
            if cur==-1 or not np.isfinite(ds[cur]):
                cur = int(np.argmax(ds)); held = 1
            else:
                best = int(np.argmax(ds))
                if best != cur and held >= 5 and ds[best] >= ds[cur] + 0.10:
                    cur = best; held = 1
                else: held += 1
            win[i] = cur
        for i, d in enumerate(idx):
            c_ix = int(win[i])
            if c_ix < 0: continue
            rzv = resz[i, c_ix]
            if not np.isfinite(rzv): continue
            tkr = cand_arr[c_ix]
            regm = regm_idx.loc[(d, tkr)] if (d, tkr) in regm_idx.index else None
            rows.append({
                'date': d, 'sector': sector, 'target': tkr,
                'residual_z': float(rzv),
                'predicted_return': float(pr[i, c_ix]) if pd.notna(pr[i, c_ix]) else 0.0,
                'next_ret': float(nxt[i, c_ix]) if pd.notna(nxt[i, c_ix]) else 0.0,
                'regime_pred': int(regm['regime_pred']) if regm is not None else 0,
                'regime_conf': float(regm['regime_conf']) if regm is not None else 0.33,
            })
    return pd.DataFrame(rows).sort_values(['date','sector']).reset_index(drop=True)

decision_panel = build_winner_panel()
print(f'decision panel: {len(decision_panel):,} rows | dates={decision_panel["date"].nunique()} | sectors={decision_panel["sector"].nunique()}')
print('\nregime_pred distribution (entire panel):')
print(decision_panel['regime_pred'].value_counts().rename(index={-1:'MR',0:'N',1:'MOM'}))

In [ ]:
# Cell 8 — PM simulator with 4 modes
PM_COST_BPS = 5.0/1e4
RZ_ENTRY    = 1.5
RZ_EXIT     = 0.30
MAX_HOLD    = 10
STOP_LOSS   = -0.02
TAKE_PROFIT =  0.03
PRED_RET_THR = 0.001

def pm_simulate(panel, mode, conf_thr=0.0,
                rz_thr=RZ_ENTRY, rz_exit=RZ_EXIT, max_hold=MAX_HOLD,
                stop=STOP_LOSS, take=TAKE_PROFIT, cost_bps=PM_COST_BPS):
    """mode in {'NoCLF','MR_GATE','NotNEUTRAL','DUAL'}.
       NoCLF      : classifier ignored.
       MR_GATE    : enter only if regm==-1 and conf>=conf_thr. Dir = fade.
       NotNEUTRAL : enter only if regm!=0 and conf>=conf_thr. Dir = fade.
       DUAL       : enter if regm!=0 and conf>=conf_thr. MR->fade; MOM->follow."""
    state = {}; rows = []
    for (date, sector), grp in panel.groupby(['date','sector'], sort=True):
        r = grp.iloc[0]
        st = state.get(sector, dict(position=0, days_in=0, trade_pnl=0.0, target=None))
        prev_pos = st['position']
        rz = float(r['residual_z']); nr = float(r['next_ret'])
        regm = int(r['regime_pred']); conf = float(r['regime_conf'])
        prr = float(r['predicted_return']) if pd.notna(r['predicted_return']) else 0.0
        desired = prev_pos; action = 'HOLD'
        if prev_pos != 0 and st['target'] != r['target']:
            desired = 0; action = 'EXIT_SWITCH'
        elif prev_pos != 0:
            st['days_in'] += 1
            if (st['days_in'] >= max_hold or abs(rz) <= rz_exit
                or st['trade_pnl'] <= stop or st['trade_pnl'] >= take):
                desired = 0; action = 'EXIT'
        if desired == 0 and abs(rz) >= rz_thr:
            ok = True; dual_follow = False
            if mode == 'NoCLF':
                ok = True
            elif mode == 'MR_GATE':
                ok = (regm == -1 and conf >= conf_thr)
            elif mode == 'NotNEUTRAL':
                ok = (regm != 0 and conf >= conf_thr)
            elif mode == 'DUAL':
                ok = (regm != 0 and conf >= conf_thr)
                dual_follow = (regm == 1)
            if ok:
                if dual_follow:  # follow MOM: rz>0 -> price rising -> LONG; rz<0 -> SHORT
                    if rz > 0 and prr > -PRED_RET_THR:   desired, action = 1, 'ENTER_LONG'
                    elif rz < 0 and prr < PRED_RET_THR:  desired, action = -1, 'ENTER_SHORT'
                else:           # fade
                    desired, action = (1, 'ENTER_LONG') if rz < 0 else (-1, 'ENTER_SHORT')
        turnover = abs(desired - prev_pos)
        gross = desired * nr; net = gross - turnover * cost_bps
        is_entry = action.startswith('ENTER'); is_exit = action.startswith('EXIT')
        if is_entry:
            st = dict(position=desired, days_in=1, trade_pnl=net, target=r['target'])
        elif is_exit:
            st = dict(position=0, days_in=0, trade_pnl=0.0, target=None)
        elif desired != 0:
            st['trade_pnl'] += net
        state[sector] = st
        rows.append({'date': date, 'sector': sector, 'action': action,
                     'position': float(desired), 'turnover': float(turnover),
                     'net_pnl': float(net),
                     'is_entry': is_entry, 'is_exit': is_exit})
    return pd.DataFrame(rows)

def portfolio_metrics(td, ppy=252):
    if td.empty: return pd.Series(dtype=float)
    daily = td.groupby('date')['net_pnl'].mean().sort_index()
    if daily.empty: return pd.Series(dtype=float)
    eq = (1+daily).cumprod(); cum = float(eq.iloc[-1]-1.0); n = len(daily)
    ann_ret = float((1+cum)**(ppy/max(n,1))-1.0)
    ann_vol = float(daily.std(ddof=1)*math.sqrt(ppy)) if n>1 else np.nan
    sh = ann_ret/ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol!=0 else np.nan
    dn = daily[daily<0]
    dnv = float(dn.std(ddof=1)*math.sqrt(ppy)) if len(dn)>1 else np.nan
    so = ann_ret/dnv if dnv and np.isfinite(dnv) and dnv!=0 else np.nan
    peak = eq.cummax(); mdd = float((eq/peak-1.0).min())
    entries = td[td['is_entry']]
    return pd.Series({
        'days': n, 'entries': int(len(entries)),
        'long':  int((entries['position']==1).sum()),
        'short': int((entries['position']==-1).sum()),
        'cum_ret': round(cum,4), 'ann_ret': round(ann_ret,4),
        'sharpe': round(sh,4) if np.isfinite(sh) else np.nan,
        'sortino': round(so,4) if np.isfinite(so) else np.nan,
        'max_dd': round(mdd,4),
        'win_rate_days': round(float((daily>0).mean()),4),
    })
print('PM ready.')

In [ ]:
# Cell 9 — Run sweep: 4 modes × {conf thresholds} on TRAIN + VAL
MODES = ['NoCLF','MR_GATE','NotNEUTRAL','DUAL']
CONF_GRID = [0.35, 0.40, 0.45, 0.55]
splits = {'TRAIN': decision_panel[decision_panel['date'].isin(train_idx)].copy(),
          'VAL':   decision_panel[decision_panel['date'].isin(val_idx)].copy()}
rows = []; t0 = time.time()
for split_name, pan in splits.items():
    for mode in MODES:
        confs = [0.0] if mode == 'NoCLF' else CONF_GRID
        for c_ in confs:
            td = pm_simulate(pan, mode=mode, conf_thr=c_)
            m  = portfolio_metrics(td)
            rows.append({'split': split_name, 'mode': mode, 'conf_thr': c_, **m.to_dict()})
summary = pd.DataFrame(rows)
summary.to_csv(DST_CACHE / 'sweep_summary.csv', index=False)
print(f'Elapsed: {time.time()-t0:.1f}s | {len(summary)} rows')

In [ ]:
# Cell 10 — Show results per split, sorted by Sharpe
cols = ['mode','conf_thr','entries','long','short','sharpe','sortino','cum_ret','max_dd','win_rate_days']
for split_name in ['TRAIN','VAL']:
    sub = summary[summary['split']==split_name].sort_values('sharpe', ascending=False)
    print(f'\n=== {split_name} — sorted by Sharpe ===')
    print(sub[cols].to_string(index=False))

In [ ]:
# Cell 11 — Edge analysis: best CLF mode vs NoCLF on VAL (filter degenerate runs)
MIN_ENTRIES = 100   # ignore degenerate variants with too few trades
val = summary[summary['split']=='VAL'].copy()
noclf_val = val[val['mode']=='NoCLF'].iloc[0]
clf_val = val[(val['mode']!='NoCLF') & (val['entries'] >= MIN_ENTRIES)].copy()
print(f'CLF candidates with entries >= {MIN_ENTRIES} on VAL:')
if clf_val.empty:
    print('  (none)')
else:
    print(clf_val[['mode','conf_thr','entries','sharpe','sortino','cum_ret','max_dd']]
          .sort_values('sharpe', ascending=False).to_string(index=False))

if clf_val.empty:
    PROMOTE = False; PROMO_MODE = 'NoCLF'; PROMO_CONF = 0.0
    print('\n>>> VERDICT: classifier too restrictive (no variant has enough trades). Keep NoCLF.')
else:
    best_clf = clf_val.sort_values('sharpe', ascending=False).iloc[0]
    delta_sh = best_clf['sharpe'] - noclf_val['sharpe']
    delta_cum = best_clf['cum_ret'] - noclf_val['cum_ret']
    print(f'\nNoCLF VAL    : Sharpe={noclf_val["sharpe"]:.4f} cum_ret={noclf_val["cum_ret"]:.4f} entries={int(noclf_val["entries"])} max_dd={noclf_val["max_dd"]:.4f}')
    print(f'Best CLF VAL : mode={best_clf["mode"]} conf={best_clf["conf_thr"]} Sharpe={best_clf["sharpe"]:.4f} cum_ret={best_clf["cum_ret"]:.4f} entries={int(best_clf["entries"])} max_dd={best_clf["max_dd"]:.4f}')
    print(f'delta Sharpe = {delta_sh:+.4f} | delta cum_ret = {delta_cum:+.4f}')
    VERDICT_THR = 0.10
    if delta_sh >= VERDICT_THR:
        print(f'\n>>> VERDICT: Classifier ADDS Sharpe edge (delta >= {VERDICT_THR}). Promote {best_clf["mode"]}@{best_clf["conf_thr"]} to TEST.')
        PROMOTE = True; PROMO_MODE = best_clf['mode']; PROMO_CONF = float(best_clf['conf_thr'])
    else:
        print(f'\n>>> VERDICT: Classifier does NOT add meaningful Sharpe edge (delta < {VERDICT_THR}). Keep NoCLF.')
        PROMOTE = False; PROMO_MODE = 'NoCLF'; PROMO_CONF = 0.0

In [ ]:
# Cell 12 — TEST the promoted config + side-by-side with NoCLF on TEST
test_pan = decision_panel[decision_panel['date'].isin(test_idx)].copy()
m_noclf = portfolio_metrics(pm_simulate(test_pan, mode='NoCLF'))
m_promo = portfolio_metrics(pm_simulate(test_pan, mode=PROMO_MODE, conf_thr=PROMO_CONF))
print(f'TEST NoCLF                    : Sharpe={m_noclf["sharpe"]:.4f} cum_ret={m_noclf["cum_ret"]:.4f} entries={int(m_noclf["entries"])} max_dd={m_noclf["max_dd"]:.4f}')
print(f'TEST {PROMO_MODE}@{PROMO_CONF}: Sharpe={m_promo["sharpe"]:.4f} cum_ret={m_promo["cum_ret"]:.4f} entries={int(m_promo["entries"])} max_dd={m_promo["max_dd"]:.4f}')

# 3-split comparison table
tr_pan = decision_panel[decision_panel['date'].isin(train_idx)].copy()
va_pan = decision_panel[decision_panel['date'].isin(val_idx)].copy()
def row(pan, mode, conf): return portfolio_metrics(pm_simulate(pan, mode=mode, conf_thr=conf))
comp = pd.DataFrame({
    'TRAIN_NoCLF': row(tr_pan, 'NoCLF', 0.0),
    'TRAIN_PROMO': row(tr_pan, PROMO_MODE, PROMO_CONF),
    'VAL_NoCLF':   row(va_pan, 'NoCLF', 0.0),
    'VAL_PROMO':   row(va_pan, PROMO_MODE, PROMO_CONF),
    'TEST_NoCLF':  m_noclf,
    'TEST_PROMO':  m_promo,
})
print('\n=== Full 3-split comparison ===')
print(comp.loc[['entries','long','short','sharpe','sortino','cum_ret','max_dd','win_rate_days']].to_string())

# log to run_history
RUN_HISTORY = Path('outputs/tuning_cache/run_history.csv')
RUN_HISTORY.parent.mkdir(parents=True, exist_ok=True)
HIST_COLS = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
             'total_entries','sharpe','sortino','cumulative_return',
             'annualized_return','annualized_volatility','max_drawdown',
             'win_rate_days','active_days']
ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
def hrow(src, tag, conf_v, mtr):
    return {'timestamp': ts, 'source': src, 'tag': tag, 'clf_trial': None, 'f1_val': None,
            'rz_thr': RZ_ENTRY, 'conf': conf_v, 'mr_exit': RZ_EXIT,
            'total_entries': int(mtr.get('entries',0)),
            'sharpe': mtr.get('sharpe'), 'sortino': mtr.get('sortino'),
            'cumulative_return': mtr.get('cum_ret'),
            'annualized_return': mtr.get('ann_ret'),
            'annualized_volatility': None,
            'max_drawdown': mtr.get('max_dd'),
            'win_rate_days': mtr.get('win_rate_days'),
            'active_days': None}
new_rows = [
    hrow('DBTS_Var_CLF_TEST_NoCLF', 'rz_x_q|all|soft|NoCLF', None, m_noclf),
    hrow('DBTS_Var_CLF_TEST_PROMO', f'rz_x_q|all|soft|{PROMO_MODE}@{PROMO_CONF}', PROMO_CONF, m_promo),
]
new_df = pd.DataFrame([{k: r.get(k) for k in HIST_COLS} for r in new_rows])
if RUN_HISTORY.exists():
    prev = pd.read_csv(RUN_HISTORY)
    combined = pd.concat([prev, new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(RUN_HISTORY, index=False)
print(f'\nLogged {len(new_df)} rows. Total history: {len(combined)}')

In [ ]:
# Cell 13 — Equity curves: NoCLF vs PROMO across the 3 splits
def equity(pan, mode, conf):
    td = pm_simulate(pan, mode=mode, conf_thr=conf)
    return (1.0 + td.groupby('date')['net_pnl'].mean().sort_index()).cumprod()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (lbl, pan) in zip(axes, [('TRAIN',tr_pan),('VAL',va_pan),('TEST',test_pan)]):
    eq_n = equity(pan, 'NoCLF', 0.0)
    eq_p = equity(pan, PROMO_MODE, PROMO_CONF)
    ax.plot(eq_n.index, eq_n.values, label='NoCLF', lw=1.6)
    ax.plot(eq_p.index, eq_p.values, label=f'{PROMO_MODE}@{PROMO_CONF}', lw=1.6, alpha=0.85)
    ax.set_title(lbl); ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=9)
    ax.set_ylabel('equity'); ax.axhline(1.0, color='black', lw=0.6, alpha=0.4)
plt.tight_layout(); plt.show()